# Intel Scene Classification with a Softmax Layer

This notebook builds a simple multiclass image classifier using one linear layer followed by softmax.
The project uses a lightweight subset of the Intel Image Classification dataset with natural scene classes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from softmax_utils import load_image_dataset, train_test_split, SoftmaxClassifier, confusion_matrix

DATA_DIR = PROJECT_ROOT / 'data' / 'intel_subset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Load the image dataset

Images are stored in one folder per class. Each image is resized to 32x32 pixels and flattened into a vector.

In [ ]:
X, y, class_names, paths = load_image_dataset(DATA_DIR, image_size=(32, 32))
print(f'Images: {len(X)}')
print(f'Classes: {class_names}')
print(f'Input features per image: {X.shape[1]}')

In [ ]:
class_counts = pd.Series([class_names[i] for i in y]).value_counts().reindex(class_names)
class_counts

## 2. Visualize sample images

In [ ]:
fig, axes = plt.subplots(len(class_names), 4, figsize=(9, 2.2 * len(class_names)))
if len(class_names) == 1:
    axes = np.array([axes])

for row, class_name in enumerate(class_names):
    class_paths = [Path(p) for p, label in zip(paths, y) if class_names[label] == class_name][:4]
    for col in range(4):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(class_paths):
            ax.imshow(Image.open(class_paths[col]).convert('RGB'))
            ax.set_title(class_name, fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Train-test split and scaling

The model uses standardized pixel values. The scaler is fitted only on the training set.

In [ ]:
X_train, X_test, y_train, y_test, train_paths, test_paths = train_test_split(X, y, paths, test_size=0.25, seed=42)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

print(f'Train images: {len(X_train)}')
print(f'Test images: {len(X_test)}')

## 4. Softmax layer

The model is a single linear layer. The output logits are converted into probabilities with softmax.

In [ ]:
model = SoftmaxClassifier(
    input_dim=X_train_scaled.shape[1],
    num_classes=len(class_names),
    seed=42
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=260,
    learning_rate=0.35,
    l2=0.002
)

history_df = pd.DataFrame(history)
history_df.tail()

## 5. Training curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.6))
ax1.plot(history_df.index + 1, history_df['loss'], color='#2E86AB', label='Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')

ax2 = ax1.twinx()
ax2.plot(history_df.index + 1, history_df['accuracy'], color='#D1495B', label='Accuracy')
ax2.set_ylabel('Training accuracy')

ax1.set_title('Softmax Training Curve')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curve.png', dpi=160)
plt.show()

## 6. Evaluate the model

In [ ]:
y_pred = model.predict(X_test_scaled)
test_accuracy = np.mean(y_pred == y_test)
print(f'Test accuracy: {test_accuracy:.2%}')

cm = confusion_matrix(y_test, y_pred, len(class_names))
fig, ax = plt.subplots(figsize=(7, 5.6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title(f'Confusion Matrix - Test Accuracy: {test_accuracy:.2%}')
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

## 7. Prediction examples

In [ ]:
probs = model.predict_proba(X_test_scaled)
confidence = probs.max(axis=1)
order = np.argsort(-confidence)[:12]

fig, axes = plt.subplots(3, 4, figsize=(10, 7.2))
for ax, idx in zip(axes.ravel(), order):
    ax.imshow(Image.open(test_paths[idx]).convert('RGB'))
    ax.axis('off')
    color = 'green' if y_pred[idx] == y_test[idx] else 'red'
    title = f'T: {class_names[y_test[idx]]}
P: {class_names[y_pred[idx]]} ({confidence[idx]:.0%})'
    ax.set_title(title, fontsize=9, color=color)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prediction_examples.png', dpi=160, bbox_inches='tight')
plt.show()

## Generated figures

![Sample images](../outputs/sample_images.png)

![Confusion matrix](../outputs/confusion_matrix.png)